In [327]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime
import os
import sys

#current_directory = os.getcwd()
sys.path.append('/Users/victoriadobryashkina/Documents/Coding/repos/risk_pylibrary')
sys.path.append('/Users/victoriadobryashkina/Documents/Coding/repos')

import tools.snowflake_db.db_connection as db
from risk_pylibrary.risk_analytics import risk_engines as re

In [328]:
## import the data, with the cut off date 31-03-2025
## sum up the values for the same instrument_ids across the books and take only one row

sql_query_data= '''
WITH ranked_instruments AS (
  SELECT 
    a.report_date, 
    a.instrument_id, 
    a.name_short,
    a.asset_type,
    sum(a.mkt_mid_eur) OVER (PARTITION BY a.report_date, a.instrument_id order by a.report_date desc) as mkt_mid_eur, 
    b.sector_name,
    a.rating_group,
    a.country,
    b.market_cap,
    a.duration_mod,
    a.current_yield,
    a.years_to_maturity,
    ROW_NUMBER() OVER (PARTITION BY a.report_date, a.instrument_id order by a.report_date desc) AS rn
  FROM TEAMS_PRD.RISK_FUNCTION_PUBLISH.pbl__risk_function_mrm_book_trading_valuation a
  LEFT JOIN TEAMS_PRD.CORE_MART.MRT_CURR__INSTRUMENTS b
    ON a.instrument_id = b.instrument_id
  WHERE a.report_date = '2025-03-31'
)

SELECT *
FROM ranked_instruments
WHERE rn = 1

'''
data = db.run_query(query=sql_query_data)

/Users/victoriadobryashkina/Documents/Coding/repos/risk_pylibrary/tools/snowflake_db/db_connection.py:181: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, CF.sf_conn)


In [329]:
def assign_tenor_bucket(years):
    if years <= 1:
        return 'T1'
    elif years <= 5:
        return 'T2'
    else:
        return 'T3'

data['tenor_bucket'] = data['years_to_maturity'].apply(assign_tenor_bucket)

In [330]:
data[data.asset_type=='BOND_CORPORATE']

,report_date,instrument_id,name_short,asset_type,mkt_mid_eur,sector_name,rating_group,country,market_cap,duration_mod,current_yield,years_to_maturity,rn,tenor_bucket
8,2025-03-31,XS2823825802,BMW Internat. Investment B.V. EO-Medium-Term N...,BOND_CORPORATE,"1,327.06","[\n """"\n]",None,NL,NaN,6.48,0.04,7.64,1,T3
22,2025-03-31,AT0000A27LQ1,voestalpine AG EO-Medium-Term Notes 2019(26),BOND_CORPORATE,"1,797.14","[\n """"\n]",None,AT,NaN,0.85,0.02,1.03,1,T2
94,2025-03-31,BE6276040431,Anheuser-Busch InBev S.A./N.V. EO-Medium-Term ...,BOND_CORPORATE,"4,292.27","[\n """"\n]",None,BE,NaN,4.61,0.02,5.05,1,T3
146,2025-03-31,XS0691970601,ÖBB-Infrastruktur AG EO-Medium-Term Notes 2011...,BOND_CORPORATE,"3,536.62","[\n """"\n]",IG,AT,NaN,1.36,0.03,1.55,1,T2
189,2025-03-31,XS1874128033,Siemens Finan.maatschappij NV EO-Medium-Term N...,BOND_CORPORATE,"3,353.19","[\n """"\n]",IG,NL,NaN,2.21,0.01,2.44,1,T2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12795,2025-03-31,XS2558395351,EnBW International Finance BV EO-Medium-Term N...,BOND_CORPORATE,"2,331.47","[\n """"\n]",IG,NL,NaN,1.44,0.04,1.65,1,T2
12967,2025-03-31,XS1074144871,"Goldman Sachs Group Inc., The EO-Med.-Term Nts...",BOND_CORPORATE,"5,173.56","[\n """"\n]",IG,US,NaN,0.99,0.03,1.18,1,T2
12987,2025-03-31,XS2630111719,Bayer AG MTN v.2023(2033/2033),BOND_CORPORATE,"3,050.41","[\n """"\n]",IG,DE,NaN,6.63,0.04,8.16,1,T3
13044,2025-03-31,XS1116263325,"Goldman Sachs Group Inc., The EO-Med.-Term Nts...",BOND_CORPORATE,"-35,870.29","[\n """"\n]",IG,US,NaN,-0.66,0.02,-0.50,1,T1


In [331]:
##total expo
print('Total Expo:', data.mkt_mid_eur.abs().sum())
##equity expo total
print('Equity Expo:',data[data.asset_type=='EQUITY'].mkt_mid_eur.abs().sum())


Total Expo: 9945090.731001522
Equity Expo: 4439941.53541109


In [332]:
## extract sectors fom braces in the data

import ast
import re

def extract_sector(text):
    try:
        cleaned = re.sub(r'\s+', ' ', text.strip())  # normalize whitespace
        parsed = ast.literal_eval(cleaned)
        return parsed[0] if isinstance(parsed, list) and parsed else None
    except Exception:
        return None

data['sector'] = data['sector_name'].apply(extract_sector)


In [333]:
## check how much of sector info is missing ( some % is expeted and considered ok, due to buckets such as "other sector" or "other idices")

def is_effectively_null(val):
    try:
        return len(val) < 2
    except:
        return True  # Treat malformed or None as null-like

# Count effectively null and non-null
effective_null_count = data['sector'].apply(is_effectively_null).sum()
effective_non_null_count = len(data) - effective_null_count

# Ratio
ratio = effective_null_count /effective_non_null_count if effective_non_null_count else float('inf')

print(f"Effective Non-Null: {effective_non_null_count}, Effective Null: {effective_null_count}, Ratio: {ratio:.2f}")


Effective Non-Null: 10638, Effective Null: 2548, Ratio: 0.24


In [334]:
# Filter for non-null-like sectors
filtered_df = data[data['sector'].apply(is_effectively_null) == False][['asset_type', 'sector']].drop_duplicates()

# Write to Excel
#filtered_df.to_excel("/Users/victoriadobryashkina/Downloads/valid_asset_type_sector.xlsx", index=False)


In [335]:
## classify the economies from the list of countries

data['country'].unique()

array(['FR', 'PT', 'US', 'JE', 'AU', 'RE', 'CA', 'NL', 'KY', 'IT', 'GR',
       'LU', 'JP', 'AT', 'IE', 'NO', 'CN', 'PL', 'BM', 'RU', 'DE', 'MX',
       'GB', '4J', 'ES', 'DK', 'BE', 'IL', 'BR', 'CH', 'LT', 'FI', 'SE',
       None, 'ID', 'AR', 'EE', 'ZA', 'TW', 'CZ', 'SG', 'TH', 'BG', 'IN',
       'HK', 'VG', 'MH', '4W', '4C', 'SK', 'CY', 'TR', 'HU', 'LI', 'GG',
       'RO', 'KR', 'UZ', 'CL', 'PE', 'CO', 'NZ', 'SI', 'PH', 'LV', 'BS',
       'PG', 'MU', 'PA', 'KZ', 'MT', 'MA', 'PR', 'IM', 'FO', 'GA', 'MC',
       'GI', '7E', 'CW', 'IS', 'AE', 'LR'], dtype=object)

In [336]:
def classify_economy_and_cap(df, country_col='country', cap_col='market_cap'):
    advanced = {
        'US', 'CA', 'GB', 'DE', 'FR', 'IT', 'ES', 'NL', 'SE', 'CH', 'AT',
        'BE', 'FI', 'IE', 'LU', 'JP', 'AU', 'NZ', 'KR', 'SG', 'DK', 'NO',
        'HK', 'IS', 'PT', 'SI', 'CY', 'EE', 'LV', 'LT', 'CZ', 'SK', 'MT',
        'BG', 'GR', 'IM', 'LI', 'JE', 'GG'
    }

    def map_country(code):
        if pd.isna(code):
            return 'Unknown'
        return 'Advanced' if code in advanced else 'Emerging'

    def map_cap_size(value):
        try:
            return 'Large Cap' if value >= 9_000_000_000 else 'Small Cap'
        except:
            return 'Unknown'

    df = df.copy()
    df['economy_type'] = df[country_col].apply(map_country)
    df['cap_size'] = df[cap_col].apply(map_cap_size)
    return df


In [337]:
# Add classifications to your portfolio DataFrame
classified_data = classify_economy_and_cap(data, country_col='country', cap_col='market_cap')


In [338]:
file_path = "/Users/victoriadobryashkina/Downloads/FRTB input/FRTB_Sector_Mapping_with_PwC_Buckets.csv"

mapping_sectors = pd.read_csv(file_path, usecols=['asset_type', 'sector', 'frtb_sector', 'pwc_bucket_group'])

print(mapping_sectors.head())

    asset_type            sector  frtb_sector  \
0       EQUITY        Immobilien  Real Estate   
1       EQUITY         Industrie  Industrials   
2       EQUITY        Halbleiter   Technology   
3  FUND_EQUITY  Mehrere Sektoren        Mixed   
4       EQUITY          Software   Technology   

                      pwc_bucket_group  
0  Financials, real estate, technology  
1      Telecommunications, industrials  
2  Financials, real estate, technology  
3                                  All  
4  Financials, real estate, technology  


In [339]:
# Perform the mapping via left join on 'sector'
merged_data = classified_data.merge(
    mapping_sectors[[ 'asset_type', 'sector','frtb_sector', 'pwc_bucket_group']],
    on=['asset_type', 'sector'],
    how='left'
)

print(merged_data.head())

  report_date instrument_id                name_short  asset_type  \
0  2025-03-31  FR0004154060                    Netgem      EQUITY   
1  2025-03-31  PTFNV1AM0002               Farminveste      EQUITY   
2  2025-03-31  US7416231022        PRIMO BRANDS CORP.      EQUITY   
3  2025-03-31  JE00B78CP782  Brent Crude Oil Lev USD   FUND_OTHER   
4  2025-03-31  AU000000RNU8       Renascor Resources       EQUITY   

   mkt_mid_eur                 sector_name rating_group country   market_cap  \
0         2.64          [\n  "Software"\n]         None      FR          NaN   
1         4.50  [\n  "Mehrere Sektoren"\n]         None      PT          NaN   
2       136.14                  [\n  ""\n]         None      US          NaN   
3       291.63          [\n  "Öl & Gas"\n]         None      JE 1,074,108.00   
4         0.15            [\n  "Mining"\n]         None      AU          NaN   

   duration_mod  current_yield  years_to_maturity  rn tenor_bucket  \
0           NaN            NaN    

In [ ]:
print('Equity Expo: ',merged_data[merged_data.asset_type.isin(['EQUITY','STOCK'])].mkt_mid_eur.abs().sum(), ' therefore must be merged together')
## 
print('Asset Type missing for Expo: ',merged_data[merged_data.asset_type.isna()].mkt_mid_eur.abs().sum(), "not too large expo-> 70% risk weight, 13 bucket equity")
## 
print('ETFs on Bonds Expo: ',merged_data[merged_data.asset_type=='FUND_BONDS'].mkt_mid_eur.abs().sum(),'not too large expo-> 12% risk weight, conservative approach corporate HY bonds, 11 bucket csr, as those are no ETFs, but a collection of bonds')
## 
print('Multi Asset ETFs Expo: ',merged_data[merged_data.asset_type=='FUND_MULTI_ASSET'].mkt_mid_eur.abs().sum(),'not too large expo-> 12% risk weight, conservative approach corporate HY bonds, 11 bucket csr')
## 
print('Other ETFs Expo: ',merged_data[merged_data.asset_type=='FUND_ARIS'].mkt_mid_eur.abs().sum(),'not too large expo-> 12% risk weight, conservative approach corporate HY bonds, 11 bucket csr')
## 
print('ETFs on Risk Free Bonds Expo: ',merged_data[merged_data.asset_type=='FUND_MONEY_MARKET'].mkt_mid_eur.abs().sum(),'ot too large expo-> 0.5% risk weight, gov bond, 1 bucket csr')
## n

Equity Expo:  5045529.104897758  therefore must be merged together
Asset Type missing for Expo:  81179.649535 not too large expo-> 70% risk weight, 13 bucket equity
ETFs on Bonds Expo:  169341.903457804 not too large expo-> 12% risk weight, conservative approach corporate HY bonds, 11 bucket csr, as those are no ETFs, but a collection of bonds
Multi Asset ETFs Expo:  4752.9320942535 not too large expo-> 12% risk weight, conservative approach corporate HY bonds, 11 bucket csr
Other ETFs Expo:  39075.91585922865 not too large expo-> 12% risk weight, conservative approach corporate HY bonds, 11 bucket csr
ETFs on Risk Free Bonds Expo:  254.29960909025002 ot too large expo-> 0.5% risk weight, gov bond, 1 bucket csr


In [341]:
merged_data.asset_type.unique()

array(['EQUITY', 'FUND_OTHER', 'BOND_CORPORATE', 'STOCK', 'FUND_EQUITY',
       'FUND_BONDS', 'BOND_GOVERNMENT', None, 'FUND_ARIS',
       'FUND_MULTI_ASSET', 'FUND_MONEY_MARKET'], dtype=object)

In [ ]:
import numpy as np

def compute_sensi(row):
    asset_type = str(row['asset_type']).upper()

    if asset_type in [
        'BOND_CORPORATE', 'BOND_GOVERNMENT', 
        'FUND_BONDS', 'FUND_MULTI_ASSET', 
        'FUND_ARIS', 'FUND_MONEY_MARKET'
    ]:
        duration = row['duration_mod'] if pd.notnull(row['duration_mod']) else 5
        #return -row['mkt_mid_eur'] * duration * 0.0001  # 1bp spread bump
        return (row['mkt_mid_eur'] * np.exp(- (row['current_yield']+0.0001) * row['years_to_maturity']) 
                -row['mkt_mid_eur'] * np.exp(- row['current_yield'] * row['years_to_maturity'])) / 0.0001  # 1bp spread bump


    elif asset_type in ['EQUITY', 'FUND_EQUITY', 'STOCK', 'FUND_OTHER']:
        return row['mkt_mid_eur']  # 1% spot bump → delta = notional

    else:
        return np.nan


# Apply the logic
merged_data['delta_sensitivity'] = merged_data.apply(compute_sensi, axis=1)

merged_data[['instrument_id', 'asset_type', 'mkt_mid_eur', 'duration_mod', 'delta_sensitivity', 'frtb_sector']].head()


,instrument_id,asset_type,mkt_mid_eur,duration_mod,delta_sensitivity,frtb_sector
0,FR0004154060,EQUITY,2.64,NaN,2.64,Technology
1,PTFNV1AM0002,EQUITY,4.50,NaN,4.50,Mixed
2,US7416231022,EQUITY,136.14,NaN,136.14,NaN
3,JE00B78CP782,FUND_OTHER,291.63,NaN,291.63,Energy and Utilities
4,AU000000RNU8,EQUITY,0.15,NaN,0.15,Materials


In [309]:
def assign_frtb_risk_weights_with_fallback(df):
    import numpy as np

    df = df.copy()

    # Equity weights (only for Advanced/Emerging + Large Cap + defined buckets)
    equity_risk_weights = {
        ## large
        ('Advanced', 'Large Cap', 'Telecommunications, industrials'): 0.35,
        ('Advanced', 'Large Cap', 'Industrials'): 0.35,
        ('Advanced', 'Large Cap', 'Consumer goods and services, transportation and storage, administrative and support service activities, healthcare, utilities'): 0.30,
        ('Advanced', 'Large Cap', 'Basic materials, energy, agriculture, manufacturing, mining and quarrying'): 0.40,
        ('Advanced', 'Large Cap', 'Financials, real estate, technology'): 0.50,
        ('Emerging', 'Large Cap', 'Telecommunications, industrials'): 0.60,
        ('Emerging', 'Large Cap', 'Consumer goods and services, transportation and storage, administrative and support service activities, healthcare, utilities'): 0.55,
        ('Emerging', 'Large Cap', 'Basic materials, energy, agriculture, manufacturing, mining and quarrying'): 0.45,
        ('Emerging', 'Large Cap', 'Financials, real estate, technology'): 0.55
    }

    # Equity FUND (index) weights 
    equity_fund_risk_weights = {
        ## large
        ('Advanced', 'Large Cap', 'Telecommunications, industrials'): 0.15,
        ('Advanced', 'Large Cap', 'Industrials'): 0.15,
        ('Advanced', 'Large Cap', 'Consumer goods and services, transportation and storage, administrative and support service activities, healthcare, utilities'): 0.15,
        ('Advanced', 'Large Cap', 'Basic materials, energy, agriculture, manufacturing, mining and quarrying'): 0.15,
        ('Advanced', 'Large Cap', 'Financials, real estate, technology'): 0.15,
        ('Advanced', 'Large Cap', 'All'): 0.25,
        ('Advanced', 'Large Cap', ''): 0.25,

    }

    # CSR weights
    csr_risk_weights = {
        ('IG', 'Sovereigns'): 0.005,
        ('IG', 'Industrials'): 0.030,
        ('IG', 'Financials'): 0.050,
        ('IG', 'Technology, telecom'): 0.020,
        ('IG', 'Healthcare, utilities'): 0.015,
        ('IG', 'Consumer/Support services'): 0.030,
        ('HY', 'Sovereigns'): 0.020,
        ('HY', 'Industrials'): 0.070,
        ('HY', 'Financials'): 0.120,
        ('HY', 'Technology, telecom'): 0.055,
        ('HY', 'Healthcare, utilities'): 0.050,
        ('HY', 'Consumer/Support services'): 0.085
    }

    def get_weight(row):
        atype = str(row.get('asset_type', '')).strip().upper()
        cap = str(row.get('cap_size', 'Large Cap')).strip()
        econ = str(row.get('economy_type', 'Advanced')).strip()
        sector_group = str(row.get('pwc_bucket_group', '')).strip()
        rating_group = str(row.get('rating_group', '')).strip()
        sector_group = ' '.join(sector_group.split())

        # EQUITY
        if atype in ['EQUITY', 'STOCK']:
            if cap == 'Small Cap':
                return 0.50 if econ == 'Advanced' else 0.70
            else:
                key = (econ, cap, sector_group)
                if key in equity_risk_weights:
                    return equity_risk_weights[key]
                else:
                    # Fallback logic for Advanced/Emerging + Large Cap + no sector
                    return 0.50 if econ == 'Advanced' else 0.60

         # EQUITY_FUND
        if atype in ['FUND_EQUITY']:
            if cap == 'Small Cap':
                return 0.70 if econ == 'Advanced' else 0.15
            else:
                key = (econ, cap, sector_group)
                if key in equity_fund_risk_weights:
                    return equity_fund_risk_weights[key]
                else:
                    # Fallback logic for Advanced/Emerging + Large Cap + no sector
                    return 0.25 if econ == 'Advanced' else 0.15
        
        # BONDS
        elif atype == 'BOND_GOVERNMENT':
            return 0.02 if econ == 'Emerging' else 0.005

        elif atype == 'BOND_CORPORATE':
            if rating_group == 'HY' or econ == 'Emerging':
                return 0.12
            else:
                return 0.05

        elif atype == 'FUND_BONDS':
            return 0.05


        # INDEX FUNDS
        elif atype in ['FUND_OTHER', 'FUND_ARIS', 'FUND_MULTI_ASSET', 'FUND_MONEY_MARKET']:
            return 0.25

        return 0.7

    df['risk_weight'] = df.apply(get_weight, axis=1)
    return df


In [310]:
current_snapshot = assign_frtb_risk_weights_with_fallback(merged_data)


In [311]:
from IPython.display import HTML

def scrollable_df(df, max_height=400, max_width=1000):
    html = df.to_html()
    styled_html = f"""
    <div style="max-height:{max_height}px; max-width:{max_width}px; overflow:auto; border:1px solid lightgray;">
        {html}
    </div>
    """
    return HTML(styled_html)


#scrollable_df(merged_data[merged_data.cap_size=='Large Cap'][['instrument_id', 'asset_type','economy_type', 'cap_size','pwc_bucket_group','risk_weight']].head(100))
scrollable_df(current_snapshot[current_snapshot.risk_weight.isnull()].drop_duplicates().head(100))





,report_date,instrument_id,name_short,asset_type,mkt_mid_eur,sector_name,rating_group,country,market_cap,duration_mod,current_yield,years_to_maturity,rn,tenor_bucket,sector,economy_type,cap_size,frtb_sector,pwc_bucket_group,delta_sensitivity,risk_weight


In [312]:
current_snapshot[['asset_type','rating_group','economy_type', 'cap_size','pwc_bucket_group','risk_weight']].drop_duplicates().sort_values(by='risk_weight').to_excel("/Users/victoriadobryashkina/Downloads/for_buckets.xlsx", index=False)



In [313]:
file_path = "/Users/victoriadobryashkina/Downloads/FRTB input/Mapped_Regulatory_Buckets.csv"

mapping_buckets = pd.read_csv(file_path, usecols=['asset_type', 'rating_group','economy_type', 'cap_size', 'pwc_bucket_group','bucket_number'])

In [314]:
# Define merge keys
merge_keys = ['asset_type', 'economy_type', 'cap_size', 'rating_group', 'pwc_bucket_group']

# Perform left merge to preserve all instruments
current_snapshot_merged = current_snapshot.merge(
    mapping_buckets[['asset_type', 'economy_type', 'cap_size', 'rating_group', 'pwc_bucket_group','bucket_number']],
    on=merge_keys,
    how='left',
    validate='m:1'  # many instruments to one bucket definition
)


In [389]:
## rearrange asset types where needed
def preprocess_asset_type_classification(df):
    """
    Standardize asset_type classification and assign risk weight and bucket_number.
    """

    df = df.copy()

    # Normalize asset_type: combine EQUITY and STOCK
    df['asset_type'] = df['asset_type'].replace({'STOCK': 'EQUITY'})

    # Assign defaults
    #df['risk_weight'] = np.nan
    #df['bucket_number'] = np.nan

    # Assign specific mappings
    df.loc[df['asset_type'].isna(), ['risk_weight', 'bucket_number', 'asset_type']] = [0.70, 13, 'EQUITY']
    df.loc[df['asset_type'] == 'FUND_BONDS', ['risk_weight', 'bucket_number']] = [0.12, 11]
    df.loc[df['asset_type'] == 'FUND_MULTI_ASSET', ['risk_weight', 'bucket_number']] = [0.12, 11]
    df.loc[df['asset_type'] == 'FUND_ARIS', ['risk_weight', 'bucket_number']] = [0.07, 12]
    df.loc[df['asset_type'] == 'FUND_MONEY_MARKET', ['risk_weight', 'bucket_number']] = [0.005, 1]

    return df
current_snapshot_merged = preprocess_asset_type_classification(current_snapshot_merged)


In [390]:
current_snapshot_merged = current_snapshot_merged.drop_duplicates()


In [391]:

(current_snapshot_merged[current_snapshot_merged.instrument_id=='AT0000A33SK7'].mkt_mid_eur * np.exp(- (current_snapshot_merged[current_snapshot_merged.instrument_id=='AT0000A33SK7'].current_yield+0.0001) * current_snapshot_merged[current_snapshot_merged.instrument_id=='AT0000A33SK7'].years_to_maturity) 
                -current_snapshot_merged[current_snapshot_merged.instrument_id=='AT0000A33SK7'].mkt_mid_eur * np.exp(- current_snapshot_merged[current_snapshot_merged.instrument_id=='AT0000A33SK7'].current_yield * current_snapshot_merged[current_snapshot_merged.instrument_id=='AT0000A33SK7'].years_to_maturity)) / 0.0001  # 1bp spread bump

13181   -35,983.55
dtype: float64

In [414]:
# Calculate weighted sensitivities (WS)
current_snapshot_merged["WS"] = current_snapshot_merged["delta_sensitivity"] * current_snapshot_merged["risk_weight"]

# Group by asset type and sum WS per asset type
current_snapshot_merged.groupby(["asset_type","bucket_number"])["WS"].sum().reset_index()



,asset_type,bucket_number,WS
0,BOND_CORPORATE,3.00,"-502,263.38"
1,BOND_CORPORATE,11.00,"-19,776.04"
2,BOND_GOVERNMENT,1.00,"-9,487.51"
3,BOND_GOVERNMENT,9.00,"-37,101.94"
4,EQUITY,1.00,"1,670.98"
5,EQUITY,2.00,737.76
6,EQUITY,3.00,258.17
7,EQUITY,4.00,"1,889.27"
8,EQUITY,5.00,"50,941.99"
9,EQUITY,6.00,"52,850.94"


In [393]:
current_snapshot_merged[(current_snapshot_merged.asset_type=='EQUITY') & (abs(current_snapshot_merged.WS)<1)].groupby("asset_type")["WS"].sum().reset_index()

,asset_type,WS
0,EQUITY,592.36


In [405]:
current_snapshot_merged[(current_snapshot_merged.asset_type=='EQUITY') & (current_snapshot_merged.WS<0)].groupby("asset_type")["WS"].count()

asset_type
EQUITY    102
Name: WS, dtype: int64

###################

In [ ]:
# Ensure tenor buckets are defined in the main dataset
df = current_snapshot_merged.copy()
df['tenor_bucket'] = df['years_to_maturity'].apply(assign_tenor_bucket)

# Compute weighted sensitivity
#df['WS'] = df['delta_sensitivity'] * df['risk_weight']
df['WS'] = df['delta_sensitivity'].fillna(0) * df['risk_weight'].fillna(0)


# Filter asset classes
#csr_assets = ['BOND_CORPORATE', 'BOND_GOVERNMENT', 'FUND_OTHER','FUND_ARIS','FUND_MULTI_ASSET','FUND_MONEY_MARKET'] ##less govies
csr_assets = ['BOND_CORPORATE', 'FUND_OTHER','FUND_ARIS','FUND_MULTI_ASSET','FUND_MONEY_MARKET']
equity_assets = ['EQUITY', 'FUND_EQUITY', 'STOCK']
csr_df = df[df['asset_type'].isin(csr_assets)].copy()
equity_df = df[df['asset_type'].isin(equity_assets)].copy()


# Updated equity correlation mapping
equity_bucket_corr = {
    1: 0.15, 2: 0.15, 3: 0.15, 4: 0.15,   # Emerging Large Cap
    5: 0.25, 6: 0.25, 7: 0.25, 8: 0.25,   # Advanced Large Cap
    9: 0.075,                             # Emerging Small Cap
    10: 0.125,                            # Advanced Small Cap
    11: 0.45,                             # Other
    12: 0.80, 13: 0.80                    # Indices
}
# Correct FRTB capital formula for CSR
def compute_correct_csr_capital(df_group):
    ws = df_group['WS'].values
    n = len(ws)
    sum_ws_squared = np.sum(ws ** 2)
    sum_cross_terms = 0.0
    for i in range(n):
        for j in range(n):
            if i != j:
                rho_name = 0.35
                rho_tenor = 1.0 if df_group.iloc[i]['tenor_bucket'] == df_group.iloc[j]['tenor_bucket'] else 0.65
                rho_basis = 1.0 if df_group.iloc[i]['country'] == df_group.iloc[j]['country'] else 0.999
                rho = rho_name * rho_tenor * rho_basis
                sum_cross_terms += rho * ws[i] * ws[j]
    return np.sqrt(max(0.0, sum_ws_squared + sum_cross_terms))

# Correct FRTB capital formula for Equity
def compute_correct_equity_capital(df_group, bucket):
    ws = df_group['WS'].values
    n = len(ws)
    sum_ws_squared = np.sum(ws ** 2)
    rho_val = equity_bucket_corr.get(int(bucket), 0.45)
    sum_cross_terms = 0.0
    for i in range(n):
        for j in range(n):
            if i != j:
                sum_cross_terms += rho_val * ws[i] * ws[j]
    return np.sqrt(max(0.0, sum_ws_squared + sum_cross_terms))

# Apply to CSR
csr_results = []
for bucket in csr_df['bucket_number'].dropna().unique():
    group = csr_df[csr_df['bucket_number'] == bucket]
    capital = compute_correct_csr_capital(group)
    csr_results.append({'bucket_number': bucket, 'capital': capital, 'asset_class': 'CSR'})

# Apply to Equity
equity_results = []
for bucket in equity_df['bucket_number'].dropna().unique():
    group = equity_df[equity_df['bucket_number'] == bucket]
    capital = compute_correct_equity_capital(group, bucket)
    equity_results.append({'bucket_number': bucket, 'capital': capital, 'asset_class': 'Equity'})

# Combine and sort results
combined_corrected_df = pd.DataFrame(csr_results + equity_results).sort_values(by=['asset_class', 'bucket_number'])
#!pip install ace_tools
#import ace_tools as tools; tools.display_dataframe_to_user(name="Corrected CSR and Equity Capital", dataframe=combined_corrected_df)


In [431]:
combined_corrected_df

,bucket_number,capital,asset_class
4,1.00,0.00,CSR
1,3.00,"277,571.52",CSR
3,11.00,"16,461.85",CSR
2,12.00,0.00,CSR
0,13.00,"4,015.04",CSR
16,1.00,944.85,Equity
15,2.00,438.68,Equity
13,3.00,138.36,Equity
12,4.00,"1,064.59",Equity
11,5.00,"26,614.30",Equity


In [433]:
print('Equity buckets sum, without correlation:',combined_corrected_df[combined_corrected_df.asset_class=="Equity"].capital.sum())
print('CSR buckets sum, without correlation:',combined_corrected_df[combined_corrected_df.asset_class=="CSR"].capital.sum())

Equity buckets sum, without correlation: 1619333.3987597148
CSR buckets sum, without correlation: 298048.40324623755


In [434]:
import pandas as pd

# Filter and inspect the 10th equity bucket from the original data
bucket_10_equity = current_snapshot_merged[
    (current_snapshot_merged['asset_type'].isin(['EQUITY', 'FUND_EQUITY', 'STOCK'])) &
    (current_snapshot_merged['bucket_number'] == 10)
].copy()

# Add derived column WS
bucket_10_equity['WS'] = bucket_10_equity['delta_sensitivity'] * bucket_10_equity['risk_weight']

# Display relevant summary
bucket_10_equity_summary = bucket_10_equity[['delta_sensitivity', 'risk_weight', 'WS']].describe()
bucket_10_equity_ws_sum = bucket_10_equity['WS'].sum()

bucket_10_equity_summary, bucket_10_equity_ws_sum


(       delta_sensitivity  risk_weight           WS
 count           7,504.00     7,762.00     7,504.00
 mean              453.57         0.50       226.79
 std            26,758.18         0.00    13,379.09
 min           -12,260.00         0.50    -6,130.00
 25%                 1.96         0.50         0.98
 50%                19.69         0.50         9.84
 75%                90.75         0.50        45.37
 max         2,315,928.42         0.50 1,157,964.21,
 np.float64(1701805.3043135216))

In [435]:
equity_df[equity_df['bucket_number'] == 10]['WS'].apply(np.sign).value_counts()


WS
1.00     7429
0.00      261
-1.00      72
Name: count, dtype: int64

In [436]:
def compute_correct_equity_capital(df_group, bucket):
    ws = df_group['WS'].values
    n = len(ws)
    sum_ws_squared = np.sum(ws ** 2)
    rho_val = equity_bucket_corr.get(int(bucket), 0.45)
    sum_cross_terms = 0.0

    for i in range(n):
        for j in range(n):
            if i != j:
                sum_cross_terms += rho_val * ws[i] * ws[j]

    # 🔍 Debug print
    print(f"Bucket {bucket} - WS²: {sum_ws_squared:.2f}, Cross: {sum_cross_terms:.2f}")

    return np.sqrt(max(0.0, sum_ws_squared + sum_cross_terms))

bucket = 10
group = equity_df[equity_df['bucket_number'] == bucket]
_ = compute_correct_equity_capital(group, bucket)



Bucket 10 - WS²: 1343423488858.56, Cross: 194089725616.32


In [437]:
equity_df[equity_df['bucket_number'] == 10]['WS'].isna().sum()


np.int64(0)

In [438]:
##why are there no WSs???
current_snapshot_merged[(current_snapshot_merged.asset_type=="EQUITY")&(current_snapshot_merged['bucket_number'] == 10) & (current_snapshot_merged['WS'].isna())]

,report_date,instrument_id,name_short,asset_type,mkt_mid_eur,sector_name,rating_group,country,market_cap,duration_mod,...,sector,economy_type,cap_size,frtb_sector,pwc_bucket_group,delta_sensitivity,risk_weight,bucket_number,WS,intra_bucket_correlation
11,2025-03-31,NL0015002AM0,"SONO GROUP NV EO -,06",EQUITY,NaN,"[\n """"\n]",None,NL,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12
45,2025-03-31,AU0000384000,FIRST GRAPHENE LTD.-ANR-,EQUITY,NaN,"[\n """"\n]",None,AU,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12
103,2025-03-31,US42237K6073,"SCORP.HLDGS.NEW DL-,0002",EQUITY,NaN,"[\n """"\n]",None,US,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12
123,2025-03-31,US8964385046,TRINITY BIOTEC.ADR NEW A,EQUITY,NaN,"[\n """"\n]",None,IE,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12
150,2025-03-31,US4976342042,KIROMIC BIOPHARMA INC.,EQUITY,NaN,"[\n """"\n]",None,US,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12911,2025-03-31,US0740142007,BEAS.BR.GRP.INC.CL.A NEW,EQUITY,NaN,"[\n """"\n]",None,US,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12
12936,2025-03-31,GG00BKTRRM22,RTW BIOTECH OPPORTUNITIES,EQUITY,NaN,"[\n """"\n]",None,GG,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12
12963,2025-03-31,US81720R6045,"SENESTECH INC. DL-,001",EQUITY,NaN,"[\n """"\n]",None,US,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12
13158,2025-03-31,US1972361026,COLUMBIA BKG SYST.,EQUITY,NaN,"[\n """"\n]",None,US,NaN,NaN,...,,Advanced,Small Cap,NaN,NaN,NaN,0.50,10.00,NaN,0.12


In [439]:
# gamma_sector generation

# Full explicit sector correlation matrix entries interpreted from FRTB table (MAR21.57)
correct_matrix_defined_pairs = {
    (1, 1): 1.0, (1, 2): 0.75, (1, 3): 0.10, (1, 4): 0.20, (1, 5): 0.25, (1, 6): 0.20, (1, 7): 0.15, (1, 8): 0.10, (1, 16): 0, (1, 17): 0.45, (1, 18): 0.45,
    (2, 2): 1.0, (2, 3): 0.05, (2, 4): 0.15, (2, 5): 0.20, (2, 6): 0.15, (2, 7): 0.10, (2, 8): 0.10, (2, 16): 0, (2, 17): 0.45, (2, 18): 0.45,
    (3, 3): 1.0, (3, 4): 0.05, (3, 5): 0.15, (3, 6): 0.20, (3, 7): 0.05, (3, 8): 0.20, (3, 16): 0, (3, 17): 0.45, (3, 18): 0.45,
    (4, 4): 1.0, (4, 5): 0.20, (4, 6): 0.25, (4, 7): 0.05, (4, 8): 0.05, (4, 16): 0, (4, 17): 0.45, (4, 18): 0.45,
    (5, 5): 1.0, (5, 6): 0.25, (5, 7): 0.05, (5, 8): 0.15, (5, 16): 0, (5, 17): 0.45, (5, 18): 0.45,
    (6, 6): 1.0, (6, 7): 0.05, (6, 8): 0.20, (5, 16): 0, (5, 17): 0.45, (5, 18): 0.45,
    (7, 7): 1.0, (7, 8): 0.05,(7, 16): 0, (7, 17): 0.45, (7, 18): 0.45,
    (8, 8): 1.0,(8, 16): 0, (8, 17): 0.45, (8, 18): 0.45,
    (17, 17): 1.0, (17, 18): 0.75,
    (18, 18): 1
}

# Ensure full symmetry by adding reverse entries
gamma_sector_fixed = {}
for (b1, b2), val in correct_matrix_defined_pairs.items():
    key = tuple(sorted((b1, b2)))
    gamma_sector_fixed[key] = val

import pandas as pd
gamma_sector_df_fixed = pd.DataFrame([
    {"bucket_1": k[0], "bucket_2": k[1], "gamma_sector": v}
    for k, v in sorted(gamma_sector_fixed.items())
])

#import ace_tools as tools
#tools.display_dataframe_to_user(name="Corrected Gamma Sector Final", dataframe=gamma_sector_df_fixed)


In [440]:
gamma_sector_fixed

{(1, 1): 1.0,
 (1, 2): 0.75,
 (1, 3): 0.1,
 (1, 4): 0.2,
 (1, 5): 0.25,
 (1, 6): 0.2,
 (1, 7): 0.15,
 (1, 8): 0.1,
 (1, 16): 0,
 (1, 17): 0.45,
 (1, 18): 0.45,
 (2, 2): 1.0,
 (2, 3): 0.05,
 (2, 4): 0.15,
 (2, 5): 0.2,
 (2, 6): 0.15,
 (2, 7): 0.1,
 (2, 8): 0.1,
 (2, 16): 0,
 (2, 17): 0.45,
 (2, 18): 0.45,
 (3, 3): 1.0,
 (3, 4): 0.05,
 (3, 5): 0.15,
 (3, 6): 0.2,
 (3, 7): 0.05,
 (3, 8): 0.2,
 (3, 16): 0,
 (3, 17): 0.45,
 (3, 18): 0.45,
 (4, 4): 1.0,
 (4, 5): 0.2,
 (4, 6): 0.25,
 (4, 7): 0.05,
 (4, 8): 0.05,
 (4, 16): 0,
 (4, 17): 0.45,
 (4, 18): 0.45,
 (5, 5): 1.0,
 (5, 6): 0.25,
 (5, 7): 0.05,
 (5, 8): 0.15,
 (5, 16): 0,
 (5, 17): 0.45,
 (5, 18): 0.45,
 (6, 6): 1.0,
 (6, 7): 0.05,
 (6, 8): 0.2,
 (7, 7): 1.0,
 (7, 8): 0.05,
 (7, 16): 0,
 (7, 17): 0.45,
 (7, 18): 0.45,
 (8, 8): 1.0,
 (8, 16): 0,
 (8, 17): 0.45,
 (8, 18): 0.45,
 (17, 17): 1.0,
 (17, 18): 0.75,
 (18, 18): 1}

In [441]:
# Merge gamma_sector with gamma_rating and compute final gamma_bc = gamma_sector * gamma_rating

# Create gamma_rating (IG: 1-8, HY: 9-15)
ig_buckets = list(range(1, 9))
hy_buckets = list(range(9, 16))
gamma_rating = {}

# Generate rating correlations
all_buckets = sorted(set(k[0] for k in gamma_sector_fixed.keys()).union(k[1] for k in gamma_sector_fixed.keys()))
for b1 in all_buckets:
    for b2 in all_buckets:
        if b1 == b2:
            continue
        key = tuple(sorted((b1, b2)))
        if key not in gamma_rating:
            gamma_rating[key] = 1.0 if (b1 in ig_buckets and b2 in ig_buckets) or (b1 in hy_buckets and b2 in hy_buckets) else 0.5

# Calculate final gamma_bc
final_gamma = {}
for key in gamma_sector_fixed:
    sector_corr = gamma_sector_fixed[key]
    rating_corr = gamma_rating.get(key, 1.0)
    final_gamma[key] = sector_corr * rating_corr

# Output as DataFrame
import pandas as pd
final_gamma_df = pd.DataFrame([
    {"bucket_1": k[0], "bucket_2": k[1], "gamma_sector": gamma_sector_fixed[k],
     "gamma_rating": gamma_rating.get(k, 1.0), "gamma_bc": final_gamma[k]}
    for k in sorted(final_gamma)
])

#import ace_tools as tools
#tools.display_dataframe_to_user(name="Final Gamma (gamma_bc) Matrix", dataframe=final_gamma_df)


In [442]:
gamma_bc = {
    (int(row['bucket_1']), int(row['bucket_2'])): row['gamma_bc']
    for _, row in final_gamma_df.iterrows()
}
# Function to compute final aggregated capital per asset class using correct FRTB inter-bucket logic
def compute_inter_bucket_aggregate(df_asset):
    bucket_cap = dict(zip(df_asset['bucket_number'], df_asset['capital']))
    buckets = list(bucket_cap.keys())

    # Diagonal part: sum of squared capital per bucket
    diagonal_sum = sum(bucket_cap[b] ** 2 for b in buckets)

    # Off-diagonal part: double sum over all unique pairs
    cross_sum = 0.0
    for i, b1 in enumerate(buckets):
        for j, b2 in enumerate(buckets):
            if b1 != b2:
                key = tuple(sorted((b1, b2)))
                gamma = gamma_bc.get(key, 0.0)
                cross_sum += gamma * bucket_cap[b1] * bucket_cap[b2]

    return np.sqrt(diagonal_sum + cross_sum)

# Separate by asset class
csr_final = combined_corrected_df[combined_corrected_df['asset_class'] == 'CSR']
equity_final = combined_corrected_df[combined_corrected_df['asset_class'] == 'Equity']

# Compute total capital per class
csr_aggregated_K = compute_inter_bucket_aggregate(csr_final)
equity_aggregated_K = compute_inter_bucket_aggregate(equity_final)

# Return results
{
    "CSR Total Capital": csr_aggregated_K,
    "Equity Total Capital": equity_aggregated_K
}


{'CSR Total Capital': np.float64(278088.22181208705),
 'Equity Total Capital': np.float64(1263845.2938308276)}